In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = SparkSession.builder.getOrCreate()

# =========================
# Dirty Customers Dataset
# =========================

customers_data = [
    (1, "John Doe", "john@example.com", "Hyderabad"),
    (2, "Alice ", "alice@example.com", "Chennai"),
    (3, None, "bob@example.com", "Bangalore"),        # NULL name
    (4, "David", None, "Mumbai"),                    # NULL email
    (5, "Eva", "eva@example.com", "Hyderabad"),
    (6, "Frank", "frank@example.com", "Delhi"),
]

customers = spark.createDataFrame(customers_data, ["customer_id", "name", "email", "city"])

# =========================
# Dirty Orders Dataset
# =========================

orders_data = [
    (101, 1, "2024-01-01", 1000),
    (102, 2, "2024-01-02", 2000),
    (103, 3, "2024-01-03", -500),     # INVALID negative value
    (104, 99, "2024-01-04", 1500),    # INVALID FK (customer_id 99)
    (105, 1, "2024-01-05", None),     # NULL amount
    (106, 5, "2024-01-06", 3000),
    (107, 5, "2024-01-07", 3000),     # duplicate-like record
]

orders = spark.createDataFrame(orders_data, ["order_id", "customer_id", "order_date", "amount"])

# =========================
# Convert date column
# =========================

orders = orders.withColumn("order_date", to_date(col("order_date")))

Practice Set A – Join Drills

In [0]:
#inner join
inner_df = orders.join(customers, "customer_id", "inner")
inner_df.show()
#left join
left_df = orders.join(customers, "customer_id", "left")
left_df.filter(col("name").isNull()).show()
#left anti join
invalid_fk = orders.join(customers, "customer_id", "left_anti")
invalid_fk.show()
#compare row counts
print("Orders count:", orders.count())
print("Inner Join count:", inner_df.count())
print("Left Join count:", left_df.count())
print("Invalid FK count:", invalid_fk.count())

+-----------+--------+----------+------+--------+-----------------+---------+
|customer_id|order_id|order_date|amount|    name|            email|     city|
+-----------+--------+----------+------+--------+-----------------+---------+
|          1|     105|2024-01-05|  NULL|John Doe| john@example.com|Hyderabad|
|          1|     101|2024-01-01|  1000|John Doe| john@example.com|Hyderabad|
|          2|     102|2024-01-02|  2000|  Alice |alice@example.com|  Chennai|
|          3|     103|2024-01-03|  -500|    NULL|  bob@example.com|Bangalore|
|          5|     107|2024-01-07|  3000|     Eva|  eva@example.com|Hyderabad|
|          5|     106|2024-01-06|  3000|     Eva|  eva@example.com|Hyderabad|
+-----------+--------+----------+------+--------+-----------------+---------+

+-----------+--------+----------+------+----+---------------+---------+
|customer_id|order_id|order_date|amount|name|          email|     city|
+-----------+--------+----------+------+----+---------------+---------+
|  

Practice Set B – Window Functions

In [0]:
#Top 3 customers per city
from pyspark.sql.window import Window

agg_df = orders.join(customers, "customer_id") .groupBy("customer_id", "name", "city").agg(sum("amount").alias("total_spend"))

window_city = Window.partitionBy("city").orderBy(col("total_spend").desc())

top3 = agg_df.withColumn("rank", rank().over(window_city)).filter(col("rank") <= 3)

top3.show()

+-----------+--------+---------+-----------+----+
|customer_id|    name|     city|total_spend|rank|
+-----------+--------+---------+-----------+----+
|          3|    NULL|Bangalore|       -500|   1|
|          2|  Alice |  Chennai|       2000|   1|
|          5|     Eva|Hyderabad|       6000|   1|
|          1|John Doe|Hyderabad|       1000|   2|
+-----------+--------+---------+-----------+----+



In [0]:
#Running total of sales
window_run = Window.orderBy("order_date")

running_total = orders.withColumn(
    "running_total",
    sum("amount").over(window_run)
)

running_total.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+--------+-----------+----------+------+-------------+
|order_id|customer_id|order_date|amount|running_total|
+--------+-----------+----------+------+-------------+
|     101|          1|2024-01-01|  1000|         1000|
|     102|          2|2024-01-02|  2000|         3000|
|     103|          3|2024-01-03|  -500|         2500|
|     104|         99|2024-01-04|  1500|         4000|
|     105|          1|2024-01-05|  NULL|         4000|
|     106|          5|2024-01-06|  3000|         7000|
|     107|          5|2024-01-07|  3000|        10000|
+--------+-----------+----------+------+-------------+



In [0]:
#Rank customers by total spend
window_rank = Window.orderBy(col("total_spend").desc())

rank_df = agg_df.withColumn("rank", rank().over(window_rank))
rank_df.show()

/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+---------+-----------+----+
|customer_id|    name|     city|total_spend|rank|
+-----------+--------+---------+-----------+----+
|          5|     Eva|Hyderabad|       6000|   1|
|          2|  Alice |  Chennai|       2000|   2|
|          1|John Doe|Hyderabad|       1000|   3|
|          3|    NULL|Bangalore|       -500|   4|
+-----------+--------+---------+-----------+----+



In [0]:
#LAG – previous order
window_lag = Window.partitionBy("customer_id").orderBy("order_date")

lag_df = orders.withColumn(
    "prev_order",
    lag("amount").over(window_lag)
)

lag_df.show()

+--------+-----------+----------+------+----------+
|order_id|customer_id|order_date|amount|prev_order|
+--------+-----------+----------+------+----------+
|     101|          1|2024-01-01|  1000|      NULL|
|     105|          1|2024-01-05|  NULL|      1000|
|     102|          2|2024-01-02|  2000|      NULL|
|     103|          3|2024-01-03|  -500|      NULL|
|     106|          5|2024-01-06|  3000|      NULL|
|     107|          5|2024-01-07|  3000|      3000|
|     104|         99|2024-01-04|  1500|      NULL|
+--------+-----------+----------+------+----------+



Practice Set C – Date Analysis

In [0]:
#1. Extract month
date_df = orders.withColumn("month", month("order_date"))
date_df.show()

+--------+-----------+----------+------+-----+
|order_id|customer_id|order_date|amount|month|
+--------+-----------+----------+------+-----+
|     101|          1|2024-01-01|  1000|    1|
|     102|          2|2024-01-02|  2000|    1|
|     103|          3|2024-01-03|  -500|    1|
|     104|         99|2024-01-04|  1500|    1|
|     105|          1|2024-01-05|  NULL|    1|
|     106|          5|2024-01-06|  3000|    1|
|     107|          5|2024-01-07|  3000|    1|
+--------+-----------+----------+------+-----+



In [0]:
#Monthly sales aggregation
monthly_sales = orders.withColumn("month", month("order_date")).groupBy("month").agg(sum("amount").alias("total_sales"))

monthly_sales.show()

+-----+-----------+
|month|total_sales|
+-----+-----------+
|    1|      10000|
+-----+-----------+



In [0]:
#Date difference
window_date = Window.partitionBy("customer_id").orderBy("order_date")

date_diff = orders.withColumn(
    "prev_date",
    lag("order_date").over(window_date)
).withColumn(
    "date_diff",
    datediff(col("order_date"), col("prev_date"))
)

date_diff.show()

+--------+-----------+----------+------+----------+---------+
|order_id|customer_id|order_date|amount| prev_date|date_diff|
+--------+-----------+----------+------+----------+---------+
|     101|          1|2024-01-01|  1000|      NULL|     NULL|
|     105|          1|2024-01-05|  NULL|2024-01-01|        4|
|     102|          2|2024-01-02|  2000|      NULL|     NULL|
|     103|          3|2024-01-03|  -500|      NULL|     NULL|
|     106|          5|2024-01-06|  3000|      NULL|     NULL|
|     107|          5|2024-01-07|  3000|2024-01-06|        1|
|     104|         99|2024-01-04|  1500|      NULL|     NULL|
+--------+-----------+----------+------+----------+---------+



In [0]:
#Trend analysis by month
trend = orders .withColumn("month", month("order_date")).groupBy("month").agg(sum("amount").alias("total_sales")).orderBy("month")

trend.show()

+-----+-----------+
|month|total_sales|
+-----+-----------+
|    1|      10000|
+-----+-----------+



Practice Set D – Timed Pipeline

In [0]:
# 1. Clean data
customers_clean = customers.dropna(subset=["name", "email"])
orders_clean = orders.filter(col("amount") > 0).dropna()

# 2. Validate referential integrity
invalid = orders_clean.join(customers_clean, "customer_id", "left_anti")
invalid.show()

# 3. Join
df = orders_clean.join(customers_clean, "customer_id")

# 4. Aggregation
agg = df.groupBy("customer_id", "name").agg(sum("amount").alias("total_spend"))

# 5. Window
window_spec = Window.orderBy(col("total_spend").desc())

final_df = agg.withColumn("rank", rank().over(window_spec))

# 6. Output
final_df.show()

+-----------+--------+----------+------+
|customer_id|order_id|order_date|amount|
+-----------+--------+----------+------+
|         99|     104|2024-01-04|  1500|
+-----------+--------+----------+------+



/databricks/python/lib/python3.12/site-packages/pyspark/sql/connect/expressions.py:1160: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


+-----------+--------+-----------+----+
|customer_id|    name|total_spend|rank|
+-----------+--------+-----------+----+
|          5|     Eva|       6000|   1|
|          2|  Alice |       2000|   2|
|          1|John Doe|       1000|   3|
+-----------+--------+-----------+----+

